In [1]:
import hmac
import hashlib
import numpy as np
import math
import random
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

## 1. Mechanisms Implementation

We encapsulate the three PQ mechanisms based on the standalone notebooks.

In [2]:
# Mechanism 1: Lattice-based STS
class LatticeSTS:
    def __init__(self, n=128):
        self.n = n
        self.q = 2 * (n**2) + 1
        self.m = 2 * n
        self.beta = int(np.sqrt(self.m))
        self.A = np.random.randint(0, self.q, (self.m, self.m))
        
    def generate_keypair_alice(self):
        s_A = np.random.randint(-self.beta//2, self.beta//2, self.m)
        p_A = (self.A @ s_A) % self.q
        return s_A, p_A
        
    def generate_keypair_bob(self):
        s_B = np.random.randint(-self.beta//2, self.beta//2, self.m)
        p_B = (s_B @ self.A) % self.q
        return s_B, p_B
        
    def compute_shared_secret_alice(self, s_A, p_B):
        val = (p_B @ s_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

    def compute_shared_secret_bob(self, s_B, p_A):
        val = (s_B @ p_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

In [3]:
# Mechanism 2: LWE/SIS Hybridization
class LWESIS:
    def __init__(self, n=64, m=128, q=10007, alpha=0.0005):
        self.n = n
        self.m = m
        self.q = q
        self.alpha = alpha
        self.M = np.random.randint(0, q, (n, m))
        
    def discrete_gaussian(self, size):
        return np.round(np.random.normal(0, self.alpha * self.q, size)).astype(int) % self.q

    def signal_function(self, y):
        return (y <= (self.q // 4)) or (y >= (3 * self.q // 4))

    def robust_extractor(self, x, sigma):
        return ((x + sigma * ((self.q - 1) // 2)) % self.q) % 2
        
    def generate_alice(self):
        s_A = np.random.randint(0, self.q, (1, self.n))
        e_A = self.discrete_gaussian((1, self.m))
        P_A = (s_A @ self.M + 2 * e_A) % self.q
        return s_A, P_A
        
    def generate_bob(self):
        s_B = self.discrete_gaussian((self.m, 1))
        P_B = (self.M @ s_B) % self.q
        return s_B, P_B

    def compute_shared_secret_alice_with_bob_sim(self, s_A, P_B, s_B, P_A):
        # A helper method to simulate the exchange
        K_A = (s_A @ P_B) % self.q
        K_B = (P_A @ s_B) % self.q
        sigma = self.signal_function(K_A[0,0])
        rk_a = self.robust_extractor(K_A[0,0], sigma)
        rk_b = self.robust_extractor(K_B[0,0], sigma)
        return hashlib.sha256(bytes([int(rk_a)])).digest(), hashlib.sha256(bytes([int(rk_b)])).digest()

In [4]:
# Mechanism 3: Module-LWE
class ModuleLWE:
    def __init__(self, n=256, q=7681):
        self.n = n
        self.q = q

    def dbl(self, x):
        e_bar = random.choices([-1, 0, 1], weights=[0.25, 0.5, 0.25])[0]
        return (2*x - e_bar) % (2*self.q)

    def cross_rounding(self, v_bar):
        return (v_bar * 2 // self.q) % 2

    def reconcile(self, v_prime, c):
        val = (2 * v_prime) % (2 * self.q)
        if c == 0:
            return 1 if ((3 * self.q) // 4 <= val < (7 * self.q) // 4) else 0
        else:
            return 1 if (val >= (5 * self.q) // 4 or val < self.q // 4) else 0
            
    def exchange(self):
        # Bob generates secret and hints
        v_bob = random.randint(0, self.q-1)
        v_bar = self.dbl(v_bob)
        c_hint = self.cross_rounding(v_bar)
        root_key_bob = (v_bar // self.q) % 2
        
        # Alice reconciles
        e_A = random.randint(-10, 10)
        v_prime = (v_bob + e_A) % self.q
        root_key_alice = self.reconcile(v_prime, c_hint)
        
        return hashlib.sha256(bytes([root_key_alice])).digest(), hashlib.sha256(bytes([root_key_bob])).digest()

## 2. User Registration (Key Bundles)

We use Lattice-based STS for Identity and Signed Pre-Keys, and LWE/SIS for ephemeral exchanges.
To support X3DH, users provide LWE/SIS compatible versions of their long-term keys.

In [5]:
lattice_sts = LatticeSTS()
lwesis = LWESIS()
module_lwe = ModuleLWE()

class PQKeyBundle:
    def __init__(self, role='alice'):
        self.role = role
        # Lattice STS Keys (IK and SPK)
        gen_func_sts = lattice_sts.generate_keypair_alice if role == 'alice' else lattice_sts.generate_keypair_bob
        self.ik_sts_priv, self.ik_sts_pub = gen_func_sts()
        self.spk_sts_priv, self.spk_sts_pub = gen_func_sts()
        
        # LWE/SIS Keys (IK, SPK, and Ephemerals)
        gen_func_lwe = lwesis.generate_alice if role == 'alice' else lwesis.generate_bob
        self.ik_lwe_priv, self.ik_lwe_pub = gen_func_lwe()
        self.spk_lwe_priv, self.spk_lwe_pub = gen_func_lwe()
        
        self.ek_lwe_privs = []
        self.ek_lwe_pubs = []
        for _ in range(10):
            priv, pub = gen_func_lwe()
            self.ek_lwe_privs.append(priv)
            self.ek_lwe_pubs.append(pub)

alice = PQKeyBundle(role='alice')
bob = PQKeyBundle(role='bob')

## 3. Extended Triple Diffie-Hellman (PQ-X3DH)

Alice computes her master secret by combining the shared secrets from multiple key exchanges.
DH1 uses Lattice-based STS.
DH2, DH3, and DH4 use LWE/SIS Hybridization.

In [6]:
# Exchange 1: IK_A and SPK_B (Lattice STS)
alice_dh_1 = lattice_sts.compute_shared_secret_alice(alice.ik_sts_priv, bob.spk_sts_pub)
bob_dh_1 = lattice_sts.compute_shared_secret_bob(bob.spk_sts_priv, alice.ik_sts_pub)

# Exchange 2: EK_A and IK_B (LWE/SIS)
alice_dh_2, bob_dh_2 = lwesis.compute_shared_secret_alice_with_bob_sim(alice.ek_lwe_privs[0], bob.ik_lwe_pub, bob.ik_lwe_priv, alice.ek_lwe_pubs[0])

# Exchange 3: EK_A and SPK_B (LWE/SIS)
alice_dh_3, bob_dh_3 = lwesis.compute_shared_secret_alice_with_bob_sim(alice.ek_lwe_privs[0], bob.spk_lwe_pub, bob.spk_lwe_priv, alice.ek_lwe_pubs[0])

# Exchange 4: EK_A and EK_B (LWE/SIS)
alice_dh_4, bob_dh_4 = lwesis.compute_shared_secret_alice_with_bob_sim(alice.ek_lwe_privs[0], bob.ek_lwe_pubs[0], bob.ek_lwe_privs[0], alice.ek_lwe_pubs[0])

alice_ms = alice_dh_1 + alice_dh_2 + alice_dh_3 + alice_dh_4
bob_ms = bob_dh_1 + bob_dh_2 + bob_dh_3 + bob_dh_4

def key_derivation_function_root(master_secret):
    return HKDF(
        algorithm=hashes.SHA256(),
        length=64,
        salt=b'\x00' * 32,
        info=b"PQ-X3DH",
    ).derive(master_secret)

alice_keys = key_derivation_function_root(master_secret=alice_ms)
alice_root_key, alice_chain_key = alice_keys[:32], alice_keys[32:]

bob_keys = key_derivation_function_root(master_secret=bob_ms)
bob_root_key, bob_chain_key = bob_keys[:32], bob_keys[32:]

print("Alice Root Key:", alice_root_key.hex())
print("Bob Root Key:  ", bob_root_key.hex())
assert alice_root_key == bob_root_key

Alice Root Key: 88e2f74b4e80ab93375267150c820735e05ab0b56928b7bf24c6157473d21ab6
Bob Root Key:   88e2f74b4e80ab93375267150c820735e05ab0b56928b7bf24c6157473d21ab6


## 4. Symmetric Ratcheting
The symmetric ratcheting relies entirely on Hash algorithms (HMAC/SHA256) which are already Post-Quantum safe.

In [7]:
def kdf_message(chain_key):
    msg_key = hmac.new(chain_key, b"\x01", hashlib.sha256).digest()
    next_chain_key = hmac.new(chain_key, b"\x02", hashlib.sha256).digest()
    return msg_key, next_chain_key

msg_key_1, alice_chain_key = kdf_message(alice_chain_key)
msg_key_2, alice_chain_key = kdf_message(alice_chain_key)

print("Message Key 1:", msg_key_1.hex())
print("Message Key 2:", msg_key_2.hex())

Message Key 1: a045cfa0fab6a92ead9d50a52fde103da8f94fa080387a31d6355e091bf237e5
Message Key 2: 9f24aa87e2a51acb65c912eaa9df4667fd5f679147f47ad155677c4b88ee5701


## 5. Asymmetric Ratcheting

For Asymmetric Ratcheting, we will use the **Module-LWE** mechanism to continually generate new forward-secret shared secrets.

In [8]:
def kdf_root(root_key, shared_secret):
    result = hmac.new(root_key, shared_secret, hashlib.sha512).digest()
    return result[:32], result[32:]

# Step: Receiver-Initiator exchange using Module-LWE
shared_secret_alice, shared_secret_bob = module_lwe.exchange()

alice_root_key, alice_chain_key = kdf_root(alice_root_key, shared_secret_alice)
bob_root_key, bob_chain_key = kdf_root(bob_root_key, shared_secret_bob)

print("New Alice Root Key:", alice_root_key.hex())
print("New Bob Root Key:  ", bob_root_key.hex())
assert alice_root_key == bob_root_key

# Step: Initiator-Receiver exchange using Module-LWE
shared_secret_alice_2, shared_secret_bob_2 = module_lwe.exchange()

alice_root_key, alice_chain_key = kdf_root(alice_root_key, shared_secret_alice_2)
bob_root_key, bob_chain_key = kdf_root(bob_root_key, shared_secret_bob_2)

print("New Alice Root Key (2):", alice_root_key.hex())
print("New Bob Root Key   (2):", bob_root_key.hex())
assert alice_root_key == bob_root_key

New Alice Root Key: 72211e58fa48f97f03396ee660c394052ab604a05f0a1a2fe085362cce4ee4fb
New Bob Root Key:   72211e58fa48f97f03396ee660c394052ab604a05f0a1a2fe085362cce4ee4fb
New Alice Root Key (2): 1fe897b0d1d70d367eed332fc05febdd1b6375c9648ed96e2bb246d66946eee1
New Bob Root Key   (2): 1fe897b0d1d70d367eed332fc05febdd1b6375c9648ed96e2bb246d66946eee1
